In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.model_selection import learning_curve, StratifiedKFold
from sklearn.svm import LinearSVC

from cso_optimizer import optimize_cso
from trainer import train_svm, evaluate

# !pip install umap-learn -q
import umap

In [ ]:
BASE_DIR = "/kaggle/working/models"
EMBEDDING_DIR = f"{BASE_DIR}/cache_embeddings"

BASELINE_MODEL_PATH = f"{BASE_DIR}/svm_baseline.joblib"
CSO_MODEL_PATH = f"{BASE_DIR}/svm_cso.joblib"
BEST_C_PATH = f"{BASE_DIR}/best_c.json"
FITNESS_CACHE_PATH = f"{BASE_DIR}/fitness_cache.json"

LABEL_MAP_PATH = "/kaggle/input/your-data/label_map.json"

In [ ]:
X_train = np.load(f"{EMBEDDING_DIR}/train_X.npy")
y_train = np.load(f"{EMBEDDING_DIR}/train_y.npy")

X_dev = np.load(f"{EMBEDDING_DIR}/dev_X.npy")
y_dev = np.load(f"{EMBEDDING_DIR}/dev_y.npy")

X_test = np.load(f"{EMBEDDING_DIR}/test_X.npy")
y_test = np.load(f"{EMBEDDING_DIR}/test_y.npy")

In [ ]:
with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_map = json.load(f)

labels = list(range(len(label_map)))
class_names = [label_map[str(i)] for i in labels]

In [ ]:
if os.path.exists(BEST_C_PATH):
    with open(BEST_C_PATH, "r", encoding="utf-8") as f:
        best_C = json.load(f)["best_C"]
    print("Loaded best_C:", best_C)
else:
    best_C = optimize_cso(
        X_train=X_train,
        y_train=y_train,
        X_dev=X_dev,
        y_dev=y_dev,
        cache_path=FITNESS_CACHE_PATH,
        P=8,
        Tmax=5,
        sample_size=100000,
        random_state=42
    )

    os.makedirs(os.path.dirname(BEST_C_PATH), exist_ok=True)
    with open(BEST_C_PATH, "w", encoding="utf-8") as f:
        json.dump({"best_C": float(best_C)}, f, ensure_ascii=False, indent=2)

    print("Saved best_C:", best_C)

In [ ]:
cso_model = train_svm(
    X_train=X_train,
    y_train=y_train,
    C=best_C,
    random_state=42
)

joblib.dump(cso_model, CSO_MODEL_PATH)
print("Saved CSO model:", CSO_MODEL_PATH)

In [ ]:
cso_dev = evaluate(cso_model, X_dev, y_dev, name="LinearSVC + CSO - Dev")
cso_test = evaluate(cso_model, X_test, y_test, name="LinearSVC + CSO - Test")

In [ ]:
y_pred_test = cso_model.predict(X_test)

In [ ]:
def plot_confusion_matrix_custom(y_true, y_pred, labels, title, normalize=True):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
        normalize="true" if normalize else None
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, cmap="viridis")

    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm[i, j]
            text = f"{value:.2f}" if normalize else str(int(value))
            ax.text(
                j, i, text,
                ha="center", va="center",
                color="white" if value > cm.max()/2 else "black",
                fontsize=8
            )

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_confusion_matrix_custom(
    y_true=y_test,
    y_pred=y_pred_test,
    labels=labels,
    title="Normalized Confusion Matrix - LinearSVC + CSO",
    normalize=True
)

In [ ]:
_, _, per_class_f1, support = precision_recall_fscore_support(
    y_test,
    y_pred_test,
    labels=labels,
    zero_division=0
)

per_class_df = pd.DataFrame({
    "label_id": labels,
    "label_name": class_names,
    "f1": per_class_f1,
    "support": support
}).sort_values("f1", ascending=False)

display(per_class_df)

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(per_class_df["label_name"], per_class_df["f1"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("F1-score")
plt.title("Per-class F1 - LinearSVC + CSO")
plt.tight_layout()
plt.show()

In [ ]:
def stratified_sample_indices(y, n_per_class=250, random_state=42):
    rng = np.random.default_rng(random_state)
    indices = []

    for label in np.unique(y):
        cls_idx = np.where(y == label)[0]
        take = min(n_per_class, len(cls_idx))
        sampled = rng.choice(cls_idx, size=take, replace=False)
        indices.extend(sampled.tolist())

    return np.array(indices)


def plot_umap(X, y, class_names, title="UMAP"):
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42
    )
    X_2d = reducer.fit_transform(X)

    plt.figure(figsize=(12, 9))

    for label in np.unique(y):
        idx = np.where(y == label)[0]
        plt.scatter(
            X_2d[idx, 0],
            X_2d[idx, 1],
            s=12,
            alpha=0.8,
            label=class_names[label]
        )

    plt.title(title, fontsize=18)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
idx_vis = stratified_sample_indices(y_test, n_per_class=250, random_state=42)
X_vis = X_test[idx_vis]
y_vis_pred = cso_model.predict(X_vis)

plot_umap(
    X_vis,
    y_vis_pred,
    class_names,
    title="UMAP - LinearSVC + CSO"
)

In [ ]:
idx_curve = stratified_sample_indices(y_train, n_per_class=2300, random_state=42)
X_curve = X_train[idx_curve]
y_curve = y_train[idx_curve]

print(X_curve.shape, y_curve.shape)

In [ ]:
def plot_learning_curve_model(estimator, X, y, title, train_sizes, cv):
    train_sizes_abs, train_scores, val_scores = learning_curve(
        estimator=estimator,
        X=X,
        y=y,
        train_sizes=train_sizes,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        shuffle=True,
        random_state=42
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)

    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    plt.figure(figsize=(10, 7))
    plt.plot(train_sizes_abs, train_mean, marker="o", label="Train score")
    plt.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.15)

    plt.plot(train_sizes_abs, val_mean, marker="o", label="CV score")
    plt.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.15)

    plt.title(title, fontsize=18)
    plt.xlabel("Training examples")
    plt.ylabel("f1_macro")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
train_sizes = np.array([3000, 10000, 17000, 24000, 30000])

cso_estimator = LinearSVC(
    C=best_C,
    max_iter=20000,
    random_state=42
)

plot_learning_curve_model(
    estimator=cso_estimator,
    X=X_curve,
    y=y_curve,
    title="Learning curve LinearSVC + CSO",
    train_sizes=train_sizes,
    cv=cv
)

In [ ]:
# compare with baseline
baseline_model = joblib.load(BASELINE_MODEL_PATH)

baseline_dev = evaluate(baseline_model, X_dev, y_dev, name="Baseline - Dev (Reload)")
baseline_test = evaluate(baseline_model, X_test, y_test, name="Baseline - Test (Reload)")

print("\n========== FINAL COMPARISON ==========")
print(f"{'Model':20s} {'C':>12s} {'Dev Macro-F1':>15s} {'Test Macro-F1':>15s} {'Test Acc':>10s}")
print("-" * 80)

print(f"{'Baseline LinearSVC':20s} "
      f"{1.0:12.6f} "
      f"{baseline_dev['macro_f1']:15.4f} "
      f"{baseline_test['macro_f1']:15.4f} "
      f"{baseline_test['accuracy']:10.4f}")

print(f"{'LinearSVC + CSO':20s} "
      f"{best_C:12.6f} "
      f"{cso_dev['macro_f1']:15.4f} "
      f"{cso_test['macro_f1']:15.4f} "
      f"{cso_test['accuracy']:10.4f}")